### RAG pipeline- data ingestion to vector db pipeline

In [48]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



In [49]:
###read all the pdfs in the data/pdf_files directory and create a list of documents
def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    #find all pdf files recursively
    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} PDF files in the directory {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader= PyPDFLoader(str(pdf_file))
            documents=loader.load()

            #add source information to metadata
            for doc in documents:
                doc.metadata['source_file']= pdf_file.name
                doc.metadata['file.type']='pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)}pages")

        except Exception as e:
            print(f"Error:{e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#process the pdf files in the data/pdf_files directory
all_pdf_documents= process_all_pdfs("../data")
    





found 1 PDF files in the directory ../data

Processing: markzuck.pdf
 Loaded 21pages

Total documents loaded: 21


In [50]:
### text splitting get into chunks

def split_documents(documents,chunk_size=600,chunk_overlap=100):
    """split document into chunks of specified size with overlap"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs= text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of split document
    if split_docs:
        print(f"\nexample chunk:")
        print(f"content: {split_docs[0].page_content[:100]}...")
        print(f"metadata: {split_docs[0].metadata}")

    return split_docs


In [51]:
chunks=split_documents(all_pdf_documents)
chunks

split 21 documents into 42 chunks

example chunk:
content: Mark Zuckerberg: A Biography...
metadata: {'producer': 'WeasyPrint 63.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdf/markzuck.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'markzuck.pdf', 'file.type': 'pdf'}


[Document(metadata={'producer': 'WeasyPrint 63.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdf/markzuck.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'markzuck.pdf', 'file.type': 'pdf'}, page_content='Mark Zuckerberg: A Biography'),
 Document(metadata={'producer': 'WeasyPrint 63.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdf/markzuck.pdf', 'total_pages': 21, 'page': 1, 'page_label': '2', 'source_file': 'markzuck.pdf', 'file.type': 'pdf'}, page_content="The Early Prodigy\nMark Zuckerberg was born in May 1984 in White Plains, New York. His early\nlife was defined by intellectual curiosity. His father, Edward, a dentist, and\nmother, Karen, a psychiatrist, fostered an environment where education was\nparamount.  Zuckerberg’s  earliest  technical  interests  were  nurtured  through\nearly home computing. He was not merely a consumer of technology but an\narchitect; his early work on 'ZuckNet' for his father's office demonstrate

### embedding and vectorstoreDB 

In [52]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity



In [53]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [54]:

from langchain_community.vectorstores import FAISS


In [55]:
print(len(chunks))
print(chunks[0])

42
page_content='Mark Zuckerberg: A Biography' metadata={'producer': 'WeasyPrint 63.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdf/markzuck.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'markzuck.pdf', 'file.type': 'pdf'}


In [61]:
print(len(chunks))


42


In [56]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9245.87it/s]


In [62]:
vectorstore.save_local("faiss_index")

In [58]:
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [59]:
vectorstore=FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

In [60]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

while True:
    query = input("You: ")

    if query.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    # retrieve relevant chunks
    docs = retriever.invoke(query)

    print("\nTop results:\n")

    for i, doc in enumerate(docs):
        print(f"Result {i+1}:")
        print(doc.page_content[:300])  # limit output
        print("-" * 50)

Exiting...
